In [1]:
import pandas as pd, numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()
books = pd.read_csv("../data/books_with_emotions.csv")
embeddings = OpenAIEmbeddings()
db = Chroma(persist_directory="../data/chroma_db", embedding_function=embeddings)

In [2]:
# pick 3 random books
selected = books.sample(3)
picks = selected["isbn13"].tolist()
selected[["title", "authors"]]

,title,authors
4240,I am smart,Suzy Capozzi
3219,Application of anti-manipulation law to EU who...,Huseyin Cagri Corlu
1734,Holding the Reins,Paisley Hope


In [3]:
descriptions = selected["description"].tolist()
vectors = embeddings.embed_documents(descriptions)
# create a new vector that averages the 3 books
avg = np.mean(vectors, axis=0).tolist()

In [4]:
# categories of the 3 entered books -- recommendations must be in this set
allowed_categories = set(selected["simple_categories"])
print("Allowed categories:", allowed_categories)

recs = db.similarity_search_by_vector(avg, k=50)   # pull more, filtering will thin it out
isbns = [int(r.page_content.strip('"').split()[0]) for r in recs]

result = books[books["isbn13"].isin(isbns)]
result = result[~result["isbn13"].isin(picks)]                          # drop the books you picked
result = result[result["simple_categories"].isin(allowed_categories)]   # keep only matching categories
result[["title", "authors", "simple_categories"]].head(10)

Allowed categories: {'Nonfiction', "Children's"}


,title,authors,simple_categories
618,The art of French kissing,Brianna R. Shrum,Children's
1187,Wolfseeker,Amy Pennza,Nonfiction
1849,Going places,Kathryn Berla,Children's
2014,Hippie Boy,Ingrid Ricks,Children's
2180,"Kate, who tamed the wind",Elizabeth Garton Scanlon,Children's
2307,Differently normal,Tammy Robinson,Children's
2563,The Knife of Never Letting Go,Patrick Ness,Children's
4012,Whispers,Greg Howard,Children's
4663,This makes me happy,Courtney Carbone,Children's
4886,Wizardmatch,Lauren Magaziner,Children's


In [5]:
def recommend_from_books(picks, k=50, n=10):
    """Recommend books based on a list of picked isbn13s.
    Averages the picks' embeddings, searches, drops the picks, and keeps
    only books whose simple_categories matches one of the picks'."""
    
    selected = books[books["isbn13"].isin(picks)]
    allowed_categories = set(selected["simple_categories"])

    avg = np.mean(embeddings.embed_documents(selected["description"].tolist()), axis=0).tolist()

    recs = db.similarity_search_by_vector(avg, k=k)   # pull more, filtering thins it out
    isbns = [int(r.page_content.strip('"').split()[0]) for r in recs]

    result = books[books["isbn13"].isin(isbns)]
    result = result[~result["isbn13"].isin(picks)]                          # drop the picks
    result = result[result["simple_categories"].isin(allowed_categories)]   # keep matching categories
    return result[["title", "authors", "simple_categories"]].head(n).reset_index(drop=True)


recommend_from_books(picks)

,title,authors,simple_categories
0,The art of French kissing,Brianna R. Shrum,Children's
1,Wolfseeker,Amy Pennza,Nonfiction
2,Going places,Kathryn Berla,Children's
3,Hippie Boy,Ingrid Ricks,Children's
4,"Kate, who tamed the wind",Elizabeth Garton Scanlon,Children's
5,Differently normal,Tammy Robinson,Children's
6,The Knife of Never Letting Go,Patrick Ness,Children's
7,Whispers,Greg Howard,Children's
8,This makes me happy,Courtney Carbone,Children's
9,Wizardmatch,Lauren Magaziner,Children's


In [6]:
# Category-coherence test: pick 3 children's books, recommendations should also be children's.
kids = books[books["simple_categories"].str.startswith("Children's")]
kids_trio = kids.sample(3, random_state=7)
kids_isbns = kids_trio["isbn13"].tolist()

print("Picked (children's):")
for t, c in zip(kids_trio["title"], kids_trio["simple_categories"]):
    print(f"  - {t}  [{c}]")

rec = recommend_from_books(kids_isbns)

share = rec["simple_categories"].str.startswith("Children's").mean()
print(f"\nChildren's share of top {len(rec)} recommendations: {share:.0%}")
rec

Picked (children's):
  - Little Do We Know  [Children's]
  - Flowers in the Attic  [Children's]
  - Camp Fire Girls amid the Snows  [Children's]

Children's share of top 5 recommendations: 100%


,title,authors,simple_categories
0,Petals on the Wind,V.C. Andrews,Children's
1,"Kate, who tamed the wind",Elizabeth Garton Scanlon,Children's
2,Only Girl in School,Natalie Standiford,Children's
3,Otherwood,Pete Hautman,Children's
4,This Heavy Silence,Nicole Mazzarella,Children's
